In [ ]:
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from src.fdm import EMechanismFDMSolver
from src.params import EMechanismFDMParams
from src.utils import generate_noisy_samples
from src.voltammetry import LinearSweepAC, LinearSweepDC

plt.style.use("seaborn-v0_8-darkgrid")

In [ ]:
dc_voltammetry = LinearSweepDC()
ac_voltammetry = LinearSweepAC()

# --- Sampling ---
key = jr.key(42)
generate_key, sampling_key, key = jr.split(key, 3)

dc_fdm_solver = EMechanismFDMSolver(dc_voltammetry)
ac_fdm_solver = EMechanismFDMSolver(ac_voltammetry)

true_params = EMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(100.0),
    E0=jnp.array(2.0),
    dB=jnp.array(1.0),
)

base_dc_current = dc_fdm_solver.solve(true_params)
base_ac_current = ac_fdm_solver.solve(true_params)

dc_samples = generate_noisy_samples(
    10,
    base_dc_current,
    0.1,
    key=key,
)

ac_samples = generate_noisy_samples(
    10,
    base_ac_current,
    0.1,
    key=key,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for sample in dc_samples:
    ax1.plot(dc_fdm_solver.applied_potentials, sample)


for sample in ac_samples:
    ax2.plot(dc_fdm_solver.applied_potentials, sample)

plt.tight_layout()
plt.show()